# Lesson 21 Lab — From Triton Source to IR and PTX

**Puzzle:** When TTIR, TTGIR, LLVM IR, PTX, and profiler correlation change together, which observation tells you whether the kernel, layout, toolchain, or hardware boundary is responsible?

This notebook retains one complete RTX 5090 execution.


## Why this matters

This lab isolates TTIR, TTGIR, LLVM IR, PTX, and profiler correlation and keeps its comparison path explicit.


## 0. Predict before running

Predict correctness, warm latency ordering, and the first boundary case. Write what would disprove each prediction.


## 1. Theory and mechanism

A Triton function passes through high-level tensor IR, target-aware GPU IR, LLVM IR, and a backend assembly stage. Reading these layers can confirm vectorization, masks, loads, and dot lowering. Performance causality still requires runtime samples and hardware counters.


## 2. Trace the mechanism

```mermaid
flowchart LR
  A["Frozen input + contract"] --> B["TTIR, TTGIR, LLVM IR, PTX, and profiler correlation"]
  B --> C["Triton candidate"]
  B --> D["CUDA / library control"]
  C --> E["correctness + samples"]
  D --> E
  E --> F["bounded decision"]
```


## 3. Inspect the comparison boundary

Baseline: documented installed capability. Candidate: reviewed Triton kernel or explicit model described below.

Spotting one expected PTX mnemonic does not prove the kernel is efficient or that it dominates application time.


## 4. Inspect the execution environment

The next cell asserts CUDA and records GPU, target, PyTorch, CUDA runtime, Triton, Python, and seed.


In [1]:
from pathlib import Path
import json, sys

ROOT = Path.cwd().parents[2]
sys.path.insert(0, str(ROOT / "scripts"))
from chapter05_runtime import environment, run_lesson

LESSON_NO = 21
LESSON_TITLE = 'From Triton Source to IR and PTX'
ENV = environment(LESSON_NO)
print(json.dumps(ENV, indent=2, ensure_ascii=False))


{
  "gpu": "NVIDIA GeForce RTX 5090",
  "compute_capability": "12.0",
  "torch": "2.13.0+cu130",
  "cuda_runtime": "13.0",
  "triton": "3.7.1",
  "triton_target": "GPUTarget(backend='cuda', arch=120, warp_size=32)",
  "python": "3.12.3",
  "seed": 20260834
}


## 5. Freeze the experiment

**Experiment:** Inspect the reviewed affine source, target identity, mask signal, and the documented dump/profiler path.

Inputs, output contract, timer, and target stay fixed across compared paths.


## 6. Inspect and execute the reviewed code

The next cell calls the shared reviewed kernel source, retains full samples in `metrics`, checks maximum error, and prints the bounded analysis.


In [2]:
metrics, analysis_en, analysis_zh = run_lesson(LESSON_NO)
print(json.dumps(metrics, indent=2, ensure_ascii=False))
print(analysis_en)


{
  "primary": 5,
  "secondary": "TTIR -> TTGIR -> LLVM IR -> PTX",
  "max_abs_error": 4.76837158203125e-07,
  "passed": true,
  "details": {
    "source_has_mask": true,
    "target": "GPUTarget(backend='cuda', arch=120, warp_size=32)",
    "inspect_commands": [
      "TRITON_KERNEL_DUMP=1",
      "proton-viewer",
      "ncu"
    ]
  }
}
The reviewed kernel source has 5 lines and targets GPUTarget(backend='cuda', arch=120, warp_size=32). IR/PTX inspection is a path to evidence, not a substitute for counters.


## 7. Read the retained RTX 5090 result

**Environment:** NVIDIA GeForce RTX 5090; compute capability 12.0; PyTorch 2.13.0+cu130; CUDA runtime 13.0; Triton 3.7.1; Python 3.12.3.

| Measured field | Checked-in value |
|---|---:|
| Source lines | 5 |
| Lowering path | TTIR -> TTGIR -> LLVM IR -> PTX |
| Maximum absolute error | 4.768e-07 |
| Acceptance gate | true |


## 8. Explain without overclaiming

The reviewed kernel source has 5 lines and targets GPUTarget(backend='cuda', arch=120, warp_size=32). IR/PTX inspection is a path to evidence, not a substitute for counters.

The installed toolchain or API surface was inspected. An available symbol or source file is not reported as native performance on an unexecuted backend.


## 9. Write the canonical artifact

The next cell stores the environment, full metrics, bilingual analysis, evidence label, and bounded conclusion.


In [3]:
artifact = Path("artifacts/rtx5090-result.json")
artifact.parent.mkdir(parents=True, exist_ok=True)
payload = {
    "lesson": LESSON_NO,
    "title": LESSON_TITLE,
    "environment": ENV,
    "evidence_label": 'compatibility-probe',
    "metrics": metrics,
    "analysis_en": analysis_en,
    "analysis_zh": analysis_zh,
    "conclusion": 'Use IR to formulate a testable hypothesis, then confirm it with a controlled kernel variant and profiler evidence.',
}
artifact.write_text(json.dumps(payload, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
print(json.dumps(payload, indent=2, ensure_ascii=False))


{
  "lesson": 21,
  "title": "From Triton Source to IR and PTX",
  "environment": {
    "gpu": "NVIDIA GeForce RTX 5090",
    "compute_capability": "12.0",
    "torch": "2.13.0+cu130",
    "cuda_runtime": "13.0",
    "triton": "3.7.1",
    "triton_target": "GPUTarget(backend='cuda', arch=120, warp_size=32)",
    "python": "3.12.3",
    "seed": 20260834
  },
  "evidence_label": "compatibility-probe",
  "metrics": {
    "primary": 5,
    "secondary": "TTIR -> TTGIR -> LLVM IR -> PTX",
    "max_abs_error": 4.76837158203125e-07,
    "passed": true,
    "details": {
      "source_has_mask": true,
      "target": "GPUTarget(backend='cuda', arch=120, warp_size=32)",
      "inspect_commands": [
        "TRITON_KERNEL_DUMP=1",
        "proton-viewer",
        "ncu"
      ]
    }
  },
  "analysis_en": "The reviewed kernel source has 5 lines and targets GPUTarget(backend='cuda', arch=120, warp_size=32). IR/PTX inspection is a path to evidence, not a substitute for counters.",
  "analysis_zh": "被审

## 10. Make the bounded decision

> Use IR to formulate a testable hypothesis, then confirm it with a controlled kernel variant and profiler evidence.

**Failure analysis:** Spotting one expected PTX mnemonic does not prove the kernel is efficient or that it dominates application time.


## 11. Extend and review

Add an awkward shape and non-contiguous layout. Stop on correctness failure. See `README.md` for references and the full review checklist.
